# Day 4 · Exercise 4: The Full Pipeline

**What you'll build:** `extract_and_validate` — all three layers connected: schema-guided prompt + JSON mode + Pydantic validation.

**Why it matters:** This is the pattern you'll use in every structured extraction task. The Pydantic model auto-generates the system prompt, JSON mode guarantees valid syntax, and `model_validate_json` enforces types. Three layers, one reliable function.

## The Schema (already defined)

In [ ]:
import ollama
import json
from pydantic import BaseModel, Field, ValidationError

MODEL = "llama3.2"

class PersonProfile(BaseModel):
    name:   str       = Field(description="Full name of the person")
    age:    int       = Field(description="Age in years")
    city:   str       = Field(description="City where they live")
    skills: list[str] = Field(description="List of professional skills")
    bio:    str       = Field(description="One-sentence biography")

## Your Implementation

In [ ]:
def extract_and_validate(text: str) -> PersonProfile:
    """Extract a PersonProfile from text using the full three-layer pipeline.

    Steps:
    1. Build a system prompt from PersonProfile.model_json_schema() using json.dumps()
    2. Call ollama.chat with format="json" and the schema-guided system prompt
    3. Pass the response string to PersonProfile.model_validate_json()
    4. Return the validated PersonProfile instance

    Args:
        text: A paragraph describing a person.

    Returns:
        A validated PersonProfile with correct types.

    Raises:
        ValidationError: if the model output doesn't match the schema.

    Example:
        profile = extract_and_validate(
            "Alice Chen, 32, is a data engineer in Cape Town "
            "who specialises in Python and SQL."
        )
        profile.name   # str: "Alice Chen"
        profile.age    # int: 32
        profile.skills # list: ["Python", "SQL"]
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _ollama_running():
    try:
        import urllib.request  # stdlib — no install needed
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

TEXT = (
    "Alice Chen is a 32-year-old data engineer based in Cape Town. "
    "She specialises in Python, SQL, and dbt. "
    "Alice builds data pipelines for financial services companies."
)

def _run_checks():
    score, total = 0, 5

    # Check 1: function exists and is callable
    try:
        assert callable(extract_and_validate), 'extract_and_validate is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: Ollama is running
    if not _ollama_running():
        print(f'{_FAIL} Check 2/{total}: Ollama server is not running')
        print('  → macOS: open the Ollama app · Linux/Windows: run ollama serve')
        return
    print(f'{_PASS} Check 2/{total}: Ollama server is reachable')
    score += 1

    # Check 3: returns a PersonProfile (not a dict)
    profile = None
    try:
        profile = extract_and_validate(TEXT)
        assert isinstance(profile, PersonProfile), \
            f'expected PersonProfile, got {type(profile).__name__} — did you call model_validate_json()?'
        print(f'{_PASS} Check 3/{total}: returned a PersonProfile instance')
        score += 1
    except ValidationError as e:
        print(f'{_FAIL} Check 3/{total}: ValidationError — {e}')
    except AssertionError as e:
        print(f'{_FAIL} Check 3/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: call failed — {e}')

    if profile is None:
        print(f'  {score}/{total} passed. Keep going!')
        return

    # Check 4: name is a str
    try:
        assert isinstance(profile.name, str), f'name should be str, got {type(profile.name).__name__}'
        assert len(profile.name) > 0, 'name is empty'
        print(f'{_PASS} Check 4/{total}: profile.name is a non-empty str ({profile.name!r})')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: age is an int
    try:
        assert isinstance(profile.age, int), \
            f'age should be int, got {type(profile.age).__name__} — Pydantic should coerce this'
        assert profile.age > 0, f'age should be positive, got {profile.age}'
        print(f'{_PASS} Check 5/{total}: profile.age is an int ({profile.age})')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 4 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Try adding a new field to `PersonProfile` — for example `years_experience: int` — and watch the prompt update automatically:

```python
class PersonProfile(BaseModel):
    name:   str       = Field(description="Full name of the person")
    age:    int       = Field(description="Age in years")
    city:   str       = Field(description="City where they live")
    skills: list[str] = Field(description="List of professional skills")
    bio:    str       = Field(description="One-sentence biography")
    years_experience: int = Field(description="Years of professional experience")

# Regenerate the schema string and call extract_and_validate again.
# The model should now return the new field without any other changes.
```

This demonstrates the "schema as single source of truth" principle.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama, json
from pydantic import ValidationError

_schema_str = json.dumps(PersonProfile.model_json_schema(), indent=2)
_example_str = (
    '{"name": "Jane Smith", "age": 28, "city": "Cape Town", '
    '"skills": ["Python", "SQL"], '
    '"bio": "A data engineer who builds data pipelines."}'
)
_SYSTEM = (
    "You are an information extractor. Read the user's text and extract "
    "the information into a JSON object with exactly these fields:\n\n"
    f"{_schema_str}\n\n"
    "Fill each field with the ACTUAL VALUE from the text — not the schema.\n"
    f"Example of a correctly filled response:\n{_example_str}\n\n"
    "Return only the JSON object — no explanation, no prose."
)

def extract_and_validate(text: str) -> PersonProfile:
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": _SYSTEM},
            {"role": "user",   "content": text},
        ],
        format="json",
    )
    return PersonProfile.model_validate_json(response["message"]["content"])
```

**Why model_validate_json instead of model_validate:** `model_validate_json(str)` parses JSON and validates in one step — it's more efficient and the right method when you have a raw JSON string from the model. `model_validate(dict)` is for when you've already parsed the JSON into a dict. Both raise `ValidationError` on bad data.

**Why add an example to the prompt:** Without an example, llama3.2 sometimes returns the JSON schema itself (the meta-description with "type", "properties" keys) instead of a filled-in instance. The example makes explicit that the model should return actual values from the text, not the schema structure.
</details>